In [1]:
!pip install tqdm boto3 requests regex sentencepiece sacremoses --quiet
!pip install transformers --quiet
!pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.2/139.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 52.6 MB/s eta 0:00:00


In [16]:
from tqdm.auto import tqdm
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
import re
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import transformers
from transformers import BertModel, BertTokenizer, get_scheduler, set_seed, AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, confusion_matrix
import pytorch_lightning as pl

from transformers import AdamW, T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [5]:
# CONFIGURATION CLASS
@dataclass
class Config:
    data_path: str = "/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Train.txt"
    test_path: str = "/content/drive/MyDrive/Dataset/Sentiment/Res_ABSA/Res_ABSA/Dev.txt"
    output_path: str = "./test.txt"
    batch_size: int = 16
    val_size: int = 500
    train_size: int = 0
    pin_memory: bool = True
    num_workers: int = 2
    seed: int = 42
    lr: float = 5e-5

In [6]:
config = Config()

In [7]:
set_seed(config.seed)

In [8]:
def readData(path):
    with open(path, 'r', encoding='utf-8') as file:
      content = file.read()
      lines = content.split('\n\n')
      data = [line.split('\n') for line in lines]
      df = pd.DataFrame(data, columns=['id', 'reviews', 'label'])
      df = df.drop(['id', 'label'], axis=1)
      def process_data(data):
          rows = []
          for entry in data:
              # Tách phần review và các cặp aspect-sentiment
              review, aspects = entry.split("\n", 1)
              # Tìm tất cả các cặp {aspect, sentiment}
              matches = re.findall(r"\{(.*?),(.*?)\}", aspects)

              # Tạo một hàng cho bảng
              row = {'review': review.strip()}
              for i, (aspect, sentiment) in enumerate(matches, start=1):
                  row[f'aspect{i}'] = aspect.strip()
                  row[f'sentiment{i}'] = sentiment.strip()

              rows.append(row)

          return rows

      # Xử lý dữ liệu
      processed_data = process_data(lines)
      dataset = pd.DataFrame(processed_data)
      dataset = dataset.drop(['review'], axis=1)
      dataset = pd.concat([df, dataset], axis=1)
      dataset = dataset.where(pd.notna(dataset), None)
    return dataset

def GenerateAspectCategoriesDict(df):
    aspectCategories = np.array(df['aspect1'])
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect2'])))
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect3'])))
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect4'])))
    aspectCategories = np.concatenate((aspectCategories, np.array(df['aspect5'])))
    aspectCategories = aspectCategories[aspectCategories != None]

    aspectCategories = aspectCategories.astype(str)
    aspectCategories = np.unique(aspectCategories)

    aspectCategoriesDictIdxToCat = dict(enumerate(aspectCategories))
    aspectCategoriesDictCatToIdx = {}
    for i, j in enumerate(aspectCategories):
        aspectCategoriesDictCatToIdx[j] = i

    return aspectCategoriesDictCatToIdx, aspectCategoriesDictIdxToCat

def PolarityDict():
    polarityDictPolToIdx = {}
    polarityDictIdxToPol = {}
    polarity = ["positive", "neutral", "negative"]
    for i, j in enumerate(polarity):
        polarityDictPolToIdx[j] = i
        polarityDictIdxToPol[i] = j
    return polarityDictPolToIdx, polarityDictIdxToPol

def GenerateAspectPolarityVector(df):
    data = np.array(df)
    aspectDictCatToIdx, aspectDictIdxToCat = GenerateAspectCategoriesDict(df)
    print(aspectDictCatToIdx)
    polarityDictCatToIdx, polarityDictIdxToCat = PolarityDict()

    categoryVec = []
    for i in range(len(df)):
        vec = [0 for k in range(36)]
        for j in range(1, 10, 2):
            if data[i][j] == None:
                break
            temp1 = aspectDictCatToIdx[data[i][j]]
            temp2 = polarityDictCatToIdx[data[i][j+1]]
            vec[(temp1*3)+temp2] = 1
        categoryVec.append(vec)

    return categoryVec

In [9]:
df = pd.DataFrame(readData(config.data_path))
df["Aspect"] = GenerateAspectPolarityVector(df)


{'AMBIENCE#GENERAL': 0, 'DRINKS#PRICES': 1, 'DRINKS#QUALITY': 2, 'DRINKS#STYLE&OPTIONS': 3, 'FOOD#PRICES': 4, 'FOOD#QUALITY': 5, 'FOOD#STYLE&OPTIONS': 6, 'LOCATION#GENERAL': 7, 'RESTAURANT#GENERAL': 8, 'RESTAURANT#MISCELLANEOUS': 9, 'RESTAURANT#PRICES': 10, 'SERVICE#GENERAL': 11}


In [10]:
df.head()

,reviews,aspect1,sentiment1,aspect2,sentiment2,aspect3,sentiment3,aspect4,sentiment4,aspect5,sentiment5,Aspect
0,Giá 53k size vừa.,DRINKS#PRICES,neutral,DRINKS#STYLE&OPTIONS,neutral,None,None,None,None,None,None,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
1,Nhưng nói chung cũng hơi đắt.,RESTAURANT#PRICES,negative,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Mình ăn rất hôi mùi dầu.,FOOD#QUALITY,negative,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Mình ăn chưa baoh thấy mùi hôi hải sản.,FOOD#QUALITY,positive,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,3 dĩa vs 2 lon Revive mà có 190k thui(.,RESTAURANT#PRICES,positive,None,None,None,None,None,None,None,None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [11]:
aspect_conversion_dict = {'RESTAURANT#GENERAL': 0, 'RESTAURANT#PRICES': 1, 'RESTAURANT#MISCELLANEOUS': 2, 'FOOD#QUALITY': 3,
    'FOOD#PRICES': 4, 'FOOD#STYLE&OPTIONS': 5, 'DRINKS#QUALITY': 6, 'DRINKS#PRICES': 7,
    'DRINKS#STYLE&OPTIONS': 8, 'LOCATION#GENERAL': 9, 'AMBIENCE#GENERAL': 10, 'SERVICE#GENERAL': 11}
sentiment_conversion_dict = {'positive': 2, 'neutral': 1, 'negative': 0}

df['aspect1'] = df['aspect1'].apply(lambda x: aspect_conversion_dict[x])
df['sentiment1'] = df['sentiment1'].apply(lambda x: sentiment_conversion_dict[x])

df['aspect2'] = df['aspect2'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
df['sentiment2'] = df['sentiment2'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

df['aspect3'] = df['aspect3'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
df['sentiment3'] = df['sentiment3'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

df['aspect4'] = df['aspect4'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
df['sentiment4'] = df['sentiment4'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

df['aspect5'] = df['aspect5'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
df['sentiment5'] = df['sentiment5'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)


In [12]:
a1 = df['aspect1']
s1 = df['sentiment1']
a1 += 1
s1 += 1
df['labels1'] = a1*s1 - 1

a2 = df['aspect2']
s2 = df['sentiment2']
a2 += 1
s2 += 1
df['labels2'] = a2*s2 - 1

df.head()

,reviews,aspect1,sentiment1,aspect2,sentiment2,aspect3,sentiment3,aspect4,sentiment4,aspect5,sentiment5,Aspect,labels1,labels2
0,Giá 53k size vừa.,8,2,9.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...",15,17.0
1,Nhưng nói chung cũng hơi đắt.,2,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,NaN
2,Mình ăn rất hôi mùi dầu.,4,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3,NaN
3,Mình ăn chưa baoh thấy mùi hôi hải sản.,4,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",11,NaN
4,3 dĩa vs 2 lon Revive mà có 190k thui(.,2,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",5,NaN


In [13]:
class T5FineTuner(pl.LightningModule):
  def __init__(self, hparams):
    super(T5FineTuner, self).__init__()
    self.hparams = hparams

    self.model = T5ForConditionalGeneration.from_pretrained(hparams.model_name_or_path)
    self.tokenizer = T5Tokenizer.from_pretrained(hparams.tokenizer_name_or_path)

  def is_logger(self):
    return self.trainer.proc_rank <= 0

  def forward(
      self, input_ids, attention_mask=None, decoder_input_ids=None, decoder_attention_mask=None, lm_labels=None
  ):
    return self.model(
        input_ids,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
        decoder_attention_mask=decoder_attention_mask,
        lm_labels=lm_labels,
    )

  def _step(self, batch):
    lm_labels = batch["target_ids"]
    lm_labels[lm_labels[:, :] == self.tokenizer.pad_token_id] = -100

    outputs = self(
        input_ids=batch["source_ids"],
        attention_mask=batch["source_mask"],
        lm_labels=lm_labels,
        decoder_attention_mask=batch['target_mask']
    )

    loss = outputs[0]

    return loss

  def training_step(self, batch, batch_idx):
    loss = self._step(batch)

    tensorboard_logs = {"train_loss": loss}
    return {"loss": loss, "log": tensorboard_logs}

  def training_epoch_end(self, outputs):
    avg_train_loss = torch.stack([x["loss"] for x in outputs]).mean()
    tensorboard_logs = {"avg_train_loss": avg_train_loss}
    return {"avg_train_loss": avg_train_loss, "log": tensorboard_logs, 'progress_bar': tensorboard_logs}

  def validation_step(self, batch, batch_idx):
    loss = self._step(batch)
    return {"val_loss": loss}

  def validation_epoch_end(self, outputs):
    avg_loss = torch.stack([x["val_loss"] for x in outputs]).mean()
    tensorboard_logs = {"val_loss": avg_loss}
    return {"avg_val_loss": avg_loss, "log": tensorboard_logs, 'progress_bar': tensorboard_logs}

  def configure_optimizers(self):
    "Prepare optimizer and schedule (linear warmup and decay)"

    model = self.model
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": self.hparams.weight_decay,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=self.hparams.learning_rate, eps=self.hparams.adam_epsilon)
    self.opt = optimizer
    return [optimizer]

  def optimizer_step(self, epoch, batch_idx, optimizer, optimizer_idx, second_order_closure=None):
    if self.trainer.use_tpu:
      xm.optimizer_step(optimizer)
    else:
      optimizer.step()
    optimizer.zero_grad()
    self.lr_scheduler.step()

  def get_tqdm_dict(self):
    tqdm_dict = {"loss": "{:.3f}".format(self.trainer.avg_loss), "lr": self.lr_scheduler.get_last_lr()[-1]}

    return tqdm_dict

  def train_dataloader(self):
    train_dataset = get_dataset(tokenizer=self.tokenizer, type_path="train", args=self.hparams)
    dataloader = DataLoader(train_dataset, batch_size=self.hparams.train_batch_size, drop_last=True, shuffle=True, num_workers=4)
    t_total = (
        (len(dataloader.dataset) // (self.hparams.train_batch_size * max(1, self.hparams.n_gpu)))
        // self.hparams.gradient_accumulation_steps
        * float(self.hparams.num_train_epochs)
    )
    scheduler = get_linear_schedule_with_warmup(
        self.opt, num_warmup_steps=self.hparams.warmup_steps, num_training_steps=t_total
    )
    self.lr_scheduler = scheduler
    return dataloader

  def val_dataloader(self):
    val_dataset = get_dataset(tokenizer=self.tokenizer, type_path="val", args=self.hparams)
    return DataLoader(val_dataset, batch_size=self.hparams.eval_batch_size, num_workers=4)

In [17]:
logger = logging.getLogger(__name__)

class LoggingCallback(pl.Callback):
  def on_validation_end(self, trainer, pl_module):
    logger.info("***** Validation results *****")
    if pl_module.is_logger():
      metrics = trainer.callback_metrics
      # Log results
      for key in sorted(metrics):
        if key not in ["log", "progress_bar"]:
          logger.info("{} = {}\n".format(key, str(metrics[key])))

  def on_test_end(self, trainer, pl_module):
    logger.info("***** Test results *****")

    if pl_module.is_logger():
      metrics = trainer.callback_metrics

      # Log and save results to file
      output_test_results_file = os.path.join(pl_module.hparams.output_dir, "test_results.txt")
      with open(output_test_results_file, "w") as writer:
        for key in sorted(metrics):
          if key not in ["log", "progress_bar"]:
            logger.info("{} = {}\n".format(key, str(metrics[key])))
            writer.write("{} = {}\n".format(key, str(metrics[key])))

In [18]:
args_dict = dict(
    model_name_or_path='t5-base',
    tokenizer_name_or_path='t5-base',
    max_seq_length=512,
    learning_rate=3e-4,
    weight_decay=0.0,
    adam_epsilon=1e-8,
    warmup_steps=0,
    train_batch_size=8,
    eval_batch_size=8,
    num_train_epochs=2,
    gradient_accumulation_steps=16,
    n_gpu=1,
    early_stop_callback=False,
    fp_16=False, # if you want to enable 16-bit training then install apex and set this to true
    opt_level='O1', # you can find out more on optimisation levels here https://nvidia.github.io/apex/amp.html#opt-levels-and-properties
    max_grad_norm=1.0, # if you enable 16-bit training then set this to a sensible value, 0.5 is a good default
    seed=42,
)

In [21]:
import argparse

args_dict.update({'num_train_epochs': 10})
args = argparse.Namespace(**args_dict)
print(args_dict)

{'model_name_or_path': 't5-base', 'tokenizer_name_or_path': 't5-base', 'max_seq_length': 512, 'learning_rate': 0.0003, 'weight_decay': 0.0, 'adam_epsilon': 1e-08, 'warmup_steps': 0, 'train_batch_size': 8, 'eval_batch_size': 8, 'num_train_epochs': 10, 'gradient_accumulation_steps': 16, 'n_gpu': 1, 'early_stop_callback': False, 'fp_16': False, 'opt_level': 'O1', 'max_grad_norm': 1.0, 'seed': 42}


In [24]:
checkpoint_callback = pl.callbacks.ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=5)

train_params = dict(
    accumulate_grad_batches=args.gradient_accumulation_steps,
    gpus=args.n_gpu,
    max_epochs=args.num_train_epochs,
    early_stop_callback=False,
    precision= 16 if args.fp_16 else 32,
    amp_level=args.opt_level,
    gradient_clip_val=args.max_grad_norm,
    checkpoint_callback=checkpoint_callback,
    callbacks=[LoggingCallback()],
)

In [25]:
class ReviewsDataset(Dataset):
    def __init__(self, train_data, tokenizer, label_col, max_sequence_len=120, as_float=False):
        self.as_float = as_float
        print("Starting Process ...")
        labels = list(train_data[label_col].values)
        # Number of exmaples.
        self.n_examples = len(labels)
        # Use tokenizer on texts. This can take a while.
        print('Using tokenizer on all texts ...')

        texts = list(train_data['reviews'].values)
        self.inputs = tokenizer(texts, add_special_tokens=False, \
                                truncation=True, padding=True, \
                                return_tensors='pt')

        # Get maximum sequence length.
        self.sequence_len = self.inputs['input_ids'].shape[-1]
        print('Texts padded or truncated to %d length!' % self.sequence_len)
        # Add labels.
        self.labels = torch.tensor(labels)
        print('Finished!\n')

    def __len__(self):
        return self.n_examples

    def __getitem__(self, i):
        if self.as_float:
            return {key: self.inputs[key][i] for key in self.inputs.keys()}, self.labels[i].to(torch.float)
        else:
            return {key: self.inputs[key][i] for key in self.inputs.keys()}, self.labels[i]

In [26]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("VietAI/vit5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("VietAI/vit5-base")
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# bert = BertModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

In [27]:
dataset = ReviewsDataset(df, tokenizer, 'labels1')

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Starting Process ...
Using tokenizer on all texts ...
Texts padded or truncated to 112 length!
Finished!



In [28]:
inputs, labels = dataset[0]
print(inputs, labels)

{'input_ids': tensor([ 4042,   134, 35879, 35808, 13636,   901, 35792,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
      

In [29]:
config.train_size = len(dataset) - config.val_size
train_ds, val_ds = random_split(dataset, [config.train_size, config.val_size])

In [30]:
train_loader = DataLoader(train_ds, config.batch_size, shuffle=True,
                          num_workers=config.num_workers,
                          pin_memory=config.pin_memory)

val_loader = DataLoader(val_ds, config.batch_size, shuffle=False,
                        num_workers=config.num_workers,
                        pin_memory=config.pin_memory)

In [31]:
for input, label in train_loader:
    print(input)
    print()
    print(label)
    break

{'input_ids': tensor([[ 1319,   781,  2341,  ...,     0,     0,     0],
        [  760,   662,   896,  ...,     0,     0,     0],
        [ 2642,  6555, 35790,  ...,     0,     0,     0],
        ...,
        [ 1449,   732,   450,  ...,     0,     0,     0],
        [ 8148,   789,   732,  ...,     0,     0,     0],
        [ 5582,  3081,  1048,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

tensor([32,  7, 20, 11, 11,  2, 10,  2, 11, 11, 11, 17, 11, 35, 11, 11])


In [32]:
def get_one_hot(outputs, k=2):
    outputs = outputs.detach()
    x = torch.topk(outputs, k)
    for i, t in enumerate(outputs):
#         x = torch.topk(t, k[i])
        for j, _ in enumerate(t):
            if j in x.indices[i]:
                t[j] = 1
            else:
                t[j] = 0
            outputs[i] = t

    outputs.requires_grad = True
    return outputs.to(torch.float)

def get_accuracy(outputs, labels):
    preds = torch.argmax(outputs, dim=1)
    return (preds == labels).sum()

def one_hot_acc(one_hot_outputs, labels):
    result = torch.all(one_hot_outputs.eq(labels))
    return result.sum()

In [33]:
model = T5FineTuner(args)

AttributeError: can't set attribute 'hparams'

In [ ]:
trainer = pl.Trainer(**train_params)

In [ ]:
trainer.fit(model)

In [ ]:
def predict(sentence):
    inputs = tokenizer(sentence, add_special_tokens=False, \
                                truncation=True, padding=True, \
                                return_tensors='pt')

    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    inputs['attention_mask'] = inputs['attention_mask'].to(device)
    inputs['input_ids'] = inputs['input_ids'].to(device)
    inputs.pop("token_type_ids", None)
    # inputs = {k: v.to(device) for k, v in inputs.items()}

    output1 = model(**inputs)
    preds1 = torch.argmax(output1, dim=1)

    # output2 = model(**inputs)
    # preds2 = torch.argmax(output2, dim=1)

    a1 = preds1.item()//3
    s1 = preds1.item()%3

    # a2 = preds2.item()//3
    # s2 = preds2.item()%3

    aspect_conversion_dict = {0: 'RESTAURANT#GENERAL', 1: 'RESTAURANT#PRICES', 2: 'RESTAURANT#MISCELLANEOUS', 3: 'FOOD#QUALITY',
    4: 'FOOD#PRICES', 5: 'FOOD#STYLE&OPTIONS', 6: 'DRINKS#QUALITY', 7: 'DRINKS#PRICES',
    8: 'DRINKS#STYLE&OPTIONS', 9: 'LOCATION#GENERAL', 10: 'AMBIENCE#GENERAL', 11: 'SERVICE#GENERAL'}
    sentiment_conversion_dict = {2: 'positive', 1: 'neutral', 0: 'negative'}

    aspect1 = aspect_conversion_dict[a1]
    sentiment1 = sentiment_conversion_dict[s1]

    # aspect2 = aspect_conversion_dict[a2]
    # sentiment2 = sentiment_conversion_dict[s2]

    prediction = {aspect1: sentiment1, }
                  # aspect2: sentiment2}

    return prediction #, a1, a2, s1, s2

In [ ]:
sentence = "Nhà hàng có không gian đẹp, nhưng chất lượng món ăn lại không như mong đợi, phục vụ hơi chậm."
# inputs = tokenizer(sentence)
# print(inputs)
prediction = predict(sentence)
print(prediction)

In [ ]:
test_df = pd.DataFrame(readData(config.test_path))
test_df.head()

In [ ]:
test_df['aspect1'] = test_df['aspect1'].apply(lambda x: aspect_conversion_dict[x])
test_df['sentiment1'] = test_df['sentiment1'].apply(lambda x: sentiment_conversion_dict[x])

test_df['aspect2'] = test_df['aspect2'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
test_df['sentiment2'] = test_df['sentiment2'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

# test_df['aspect3'] = test_df['aspect3'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
# test_df['sentiment3'] = test_df['sentiment3'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

# test_df['aspect4'] = test_df['aspect4'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
# test_df['sentiment4'] = test_df['sentiment4'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

# df['aspect5'] = df['aspect5'].apply(lambda x: aspect_conversion_dict[x] if x is not None else None)
# df['sentiment5'] = df['sentiment5'].apply(lambda x: sentiment_conversion_dict[x] if x is not None else None)

a1 = test_df['aspect1']
s1 = test_df['sentiment1']
a1 += 1
s1 += 1
test_df['labels1'] = a1*s1 - 1

# a2 = test_df['aspect2']
# s2 = test_df['sentiment2']
# a2 += 1
# s2 += 1
# test_df['labels2'] = a2*s2 - 1

In [ ]:
model.model.eval()
outputs = []
targets = []
for batch in tqdm(loader):
  outs = model.model.generate(input_ids=batch['source_ids'].cuda(),
                              attention_mask=batch['source_mask'].cuda(),
                              max_length=200)

  dec = [tokenizer.decode(ids) for ids in outs]
  target = [tokenizer.decode(ids) for ids in batch["target_ids"]]

  outputs.extend(dec)
  targets.extend(target)

In [ ]:
# Hàm test
def test(data_loader):
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")

    predicts = []
    real_values = []

    for inputs, label in data_loader:
        inputs['input_ids'] = inputs['input_ids'].to(device)
        inputs.pop("token_type_ids", None)
        inputs['attention_mask'] = inputs['attention_mask'].to(device)

        outputs = model(**inputs)

        label = label.to(torch.long)
        label = label.to(device)

        # acc = get_accuracy(outputs, label)

        _, pred = torch.max(outputs, dim=1)
        predicts.extend(pred)
        real_values.extend(label)

    predicts = torch.stack(predicts).cpu()
    real_values = torch.stack(real_values).cpu()
    print(classification_report(real_values, predicts))
    return real_values, predicts

In [ ]:
data_test = ReviewsDataset(test_df, tokenizer, label_col='labels1')
test_loader = DataLoader(data_test, config.batch_size, shuffle=True,
                          num_workers=config.num_workers,
                          pin_memory=config.pin_memory)

In [ ]:
real_values, predicts = test(test_loader)

#Plotting results

In [ ]:
def plot(history, name="HistoryPlot", figsize=(20, 9)):
    fig = plt.figure(figsize=figsize)
    epochs = [x['epoch'] for x in history]

    # Plotting Losses
    ax1 = fig.add_subplot(121)
    ax1.locator_params(axis='x', nbins=5)
    train_losses = [x['train_loss'] for x in history]
    val_losses = [x['val_loss'] for x in history]
    ax1.plot(epochs, train_losses, label='Train-Losses')
    ax1.plot(epochs, val_losses, label='Validation-Losses')
    plt.xlabel('Epochs')
    plt.ylabel('Losses')
    plt.title('Losses vs Epochs')
    plt.legend()

    # Plotting Accuracies
    ax2 = fig.add_subplot(122)
    ax2.locator_params(axis='x', nbins=5)
    train_accs = [x['train_acc'].cpu() for x in history]
    val_accs = [x['val_acc'].cpu() for x in history]
    ax2.plot(epochs, train_accs, label='Train-Accuracies')
    ax2.plot(epochs, val_accs, label='Validation-Accuracies')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracies')
    plt.title('Accuracies vs Epochs')
    plt.legend()

    fig.savefig('./'+name+".jpg")
    plt.show()

In [ ]:
plot(history, name="HistoryPlot1")